# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Loading `fact_content_daily_performance` for March 2026 (same slice as ML-04/ML-06/ML-07/
ML-08) and aggregating to one row per page. No categorical columns exist in this table
(no content type, no intent tag), so the feature vector here is purely numeric aggregates.

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from huggingface_hub import login, notebook_login

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    notebook_login()
    HF_TOKEN = os.environ.get("HF_TOKEN")

data_files = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
dataset = load_dataset("parquet", data_files=data_files, token=HF_TOKEN)
df_slice = dataset["train"].to_pandas()
df_slice["report_date"] = pd.to_datetime(df_slice["report_date"])

print(f"Loaded {len(df_slice):,} rows for March 2026")

# Aggregate to one row per page
page = df_slice.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
    days_with_data=("report_date", "nunique"),
).reset_index()

# Engineered feature: CTR, with a safe fill for zero-impression pages
page["ctr"] = np.where(page["impressions"] > 0, page["clicks"] / page["impressions"], 0)

# Fill any remaining missing values (e.g. avg_position can be NaN if a page never ranked)
page["avg_position"] = page["avg_position"].fillna(0)

# Label: first-half vs second-half clicks within the month
median_date = df_slice["report_date"].median()
first_clicks = df_slice[df_slice["report_date"] < median_date].groupby("content_hash_id")["gsc_clicks"].sum()
second_clicks = df_slice[df_slice["report_date"] >= median_date].groupby("content_hash_id")["gsc_clicks"].sum()
page["first_half_clicks"] = page["content_hash_id"].map(first_clicks).fillna(0)
page["second_half_clicks"] = page["content_hash_id"].map(second_clicks).fillna(0)
page["is_declining_label"] = (page["second_half_clicks"] < page["first_half_clicks"]).astype(int)

print(f"Feature vector built: {len(page):,} pages, {page.shape[1]} columns")
display(page.head())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 9,841,378 rows for March 2026
Feature vector built: 331,437 pages, 10 columns


,content_hash_id,client_hash_id,impressions,clicks,avg_position,days_with_data,ctr,first_half_clicks,second_half_clicks,is_declining_label
0,content_000005d4ced12088,client_9958f0a7ae1df715,86,0,72.854861,31,0.0,0.0,0.0,0
1,content_00001e488b74b799,client_625b6439094e23e4,0,0,0.000000,31,0.0,0.0,0.0,0
2,content_00007bd2985b77c3,client_73cda7b4e4f265ea,47,0,5.269565,31,0.0,0.0,0.0,0
3,content_00008950670cb6b5,client_def0955f7a377868,0,0,0.000000,31,0.0,0.0,0.0,0
4,content_0000a348850eb1fc,client_3ffa76342f366962,0,0,0.000000,29,0.0,0.0,0.0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists
BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before decision moment? |
|---|---|---|---|
| `impressions` | Total GSC impressions, whole month | Sum defaults to 0 if no rows | Yes — whole-month aggregate, known at review time |
| `clicks` | Total GSC clicks, whole month | Sum defaults to 0 if no rows | Yes |
| `avg_position` | Mean search position, whole month | Filled with 0 if page never ranked | Yes |
| `ctr` | Engineered: clicks / impressions | 0 if impressions = 0 (avoids divide-by-zero) | Yes — derived from the two fields above |
| `days_with_data` | Count of distinct report_date rows with data | No fill needed, always ≥ 0 | Yes — whole-month count |

No categorical columns exist in `fact_content_daily_performance` (unlike the starter CSV's
`content_type`/`main_intent`), so there's no one-hot encoding needed for this table.

**Excluded on purpose:** `first_half_clicks` and `second_half_clicks` are used only to build
`is_declining_label` — they are NOT available before the decision moment in the same sense
as the other features, since `second_half_clicks` is literally half of what defines the
label. They are dropped from the feature set in Section 3.

In [ ]:
feature_cols = ["impressions", "clicks", "avg_position", "ctr", "days_with_data"]
print("Feature vector columns:", feature_cols)
print("\nMissing value check:")
display(page[feature_cols].isnull().sum())

Feature vector columns: ['impressions', 'clicks', 'avg_position', 'ctr', 'days_with_data']

Missing value check:


,0
impressions,0
clicks,0
avg_position,0
ctr,0
days_with_data,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the
test.*

I deliberately add `second_half_clicks` back in as a feature — the exact column that defines
half of the label — and watch the score jump toward perfect. Then I remove it and confirm the
honest number.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

RANDOM_STATE = 42
target = "is_declining_label"

X_honest = page[feature_cols]
X_leaky = page[feature_cols + ["second_half_clicks"]]
y = page[target]

X_train_h, X_test_h, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_l, X_test_l, _, _ = train_test_split(
    X_leaky, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

leaky_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
leaky_model.fit(X_train_l, y_train)
leaky_preds = leaky_model.predict(X_test_l)
leaky_probs = leaky_model.predict_proba(X_test_l)[:, 1]

honest_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
honest_model.fit(X_train_h, y_train)
honest_preds = honest_model.predict(X_test_h)
honest_probs = honest_model.predict_proba(X_test_h)[:, 1]

print("--- LEAKY MODEL (second_half_clicks included as a feature) ---")
print(f"ROC-AUC: {roc_auc_score(y_test, leaky_probs):.4f} (artificially inflated)")
print(f"F1-Score: {f1_score(y_test, leaky_preds):.4f}")

print("\n--- HONEST MODEL (second_half_clicks removed) ---")
print(f"ROC-AUC: {roc_auc_score(y_test, honest_probs):.4f} (realistic generalization)")
print(f"F1-Score: {f1_score(y_test, honest_preds):.4f}")

print(f"\nROC-AUC inflation from the leak: {roc_auc_score(y_test, leaky_probs) - roc_auc_score(y_test, honest_probs):+.4f}")
print("Confirmed: my final feature set for ML-08 onward uses only the honest columns.")

--- LEAKY MODEL (second_half_clicks included as a feature) ---
ROC-AUC: 1.0000 (artificially inflated)
F1-Score: 0.9918

--- HONEST MODEL (second_half_clicks removed) ---
ROC-AUC: 0.9459 (realistic generalization)
F1-Score: 0.4608

ROC-AUC inflation from the leak: +0.0540
Confirmed: my final feature set for ML-08 onward uses only the honest columns.


## 4. What I excluded and why

The list of fields I refused to use as features, with one line of why each:

- **`content_hash_id`, `client_hash_id`** — identifiers, used only for joining/grouping
  (client-holdout splits), never as a model input.
- **`report_date`** — a raw date isn't a feature; the day-level rows are aggregated to
  whole-month signals instead, so no single date can leak.
- **`first_half_clicks`, `second_half_clicks`** — these directly construct
  `is_declining_label`. Including either would leak the label into the features, as shown in
  Section 3.
- **Any FlyRank product flag (health_score, is_quick_win, etc.)** — not present in
  `fact_content_daily_performance` by design, and would not be used even if available, since
  the goal is to learn from observable evidence, not the product's own decision.
- **Titles, URLs, keyword text** — not present in this table; would never be committed to a
  public repo even if they were.

In [ ]:
excluded_fields = [
    "content_hash_id (identifier, grouping only)",
    "client_hash_id (identifier, grouping only)",
    "report_date (raw date, aggregated instead)",
    "first_half_clicks (label-derived)",
    "second_half_clicks (label-derived, proven leaky above)",
    "FlyRank product flags (not in this table by design)",
]
print("Excluded from feature set:")
for field in excluded_fields:
    print(f"  - {field}")

print("\nFinal honest feature set carried forward to ML-08:", feature_cols)

Excluded from feature set:
  - content_hash_id (identifier, grouping only)
  - client_hash_id (identifier, grouping only)
  - report_date (raw date, aggregated instead)
  - first_half_clicks (label-derived)
  - second_half_clicks (label-derived, proven leaky above)
  - FlyRank product flags (not in this table by design)

Final honest feature set carried forward to ML-08: ['impressions', 'clicks', 'avg_position', 'ctr', 'days_with_data']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.